In [2]:
import numpy as np
import Scope
from Scope import *
import matplotlib.pyplot as plt
from rdkit.Chem import Draw
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem import Descriptors, rdMolDescriptors
from sklearn.decomposition import PCA
from rdkit import Chem
import mols2grid

In [4]:
#systems = load_binary("finished_systems.npy")

In [5]:
systems_homoleptic = []
systems_homoleptic_id = []
for idx, system in enumerate(systems):
    ligands = system.HS_ref_mol.ligands
    if all(compare_species(ligand,ligands[0]) for ligand in ligands) or ligands[0].get_denticity()==6:  # Ensure all ligands are the same           
        systems_homoleptic.append(system.HS_ref_mol) # Attention for compare
        systems_homoleptic_id.append(idx)
print(len(systems_homoleptic))
print(systems_homoleptic[0].ligands)

388
[------------- SCOPE LIGAND Object --------------
 Version               = 2.0
 Type                  = specie
 Sub-Type              = ligand
 Number of Atoms       = 26
 Formula               = H11-C11-N3-O
 Has Adjacency Matrix  = YES
 Number of Parents     = 2
 Total Charge          = 0
 SMILES                = [H]C(=Nc1c([H])c([H])c(OC([H])([H])[H])c([H])c1[H])c1nc([H])c([H])n1[H]
 # HAS RDKIT OBJECT    = YES
------------------------------------------------
, ------------- SCOPE LIGAND Object --------------
 Version               = 2.0
 Type                  = specie
 Sub-Type              = ligand
 Number of Atoms       = 26
 Formula               = H11-C11-N3-O
 Has Adjacency Matrix  = YES
 Number of Parents     = 2
 Total Charge          = 0
 SMILES                = [H]C(=Nc1c([H])c([H])c(OC([H])([H])[H])c([H])c1[H])c1nc([H])c([H])n1[H]
 # HAS RDKIT OBJECT    = YES
------------------------------------------------
, ------------- SCOPE LIGAND Object --------------
 Version  

In [6]:
def compute_pca(fingerprints, n, debug: int = 0):
    # If debug mode is enabled (debug > 0), display the size of the input array
    if debug > 0:
        print("Size of fingerprints array:", len(fingerprints))

    # Check if the input list is not empty
    if fingerprints:
        # Initialize PCA with the specified number of components
        pca = PCA(n_components=n)
        
        # Perform PCA on the input data (convert to NumPy array first)
        pca_coords = pca.fit_transform(np.array(fingerprints))
        
        # Retrieve the variance explained by each component
        vars = pca.explained_variance_ratio_

        # Print the cumulative variance explained by the first n components
        print(np.sum(vars[0:n]))

        # Compute the Euclidean norm of each transformed point (in PCA space)
        # Round the result to 10 decimal places
        return np.round(np.linalg.norm(pca_coords, axis=1), 10)

    # Return an empty list if input is empty
    return []

In [7]:
def compute_pca_descriptors(systems_homoleptic, n, debug: int = 0):
    systems_homoleptic_aro = []

    # Step 1: Check aromaticity and extract Murcko Scaffold
    for i in range(len(systems_homoleptic)):
        # Extract the first SMILES string from the current system
        smiles_str = systems_homoleptic[i].smiles[0]
        
        # Convert the SMILES string to a molecule object
        mol = Chem.MolFromSmiles(smiles_str)

        # Proceed only if the molecule is valid
        if mol:
            # Determine if the molecule is aromatic by checking all atoms
            aromatic_atoms = [atom.GetIsAromatic() for atom in mol.GetAtoms()]
            is_aromatic = any(aromatic_atoms)  # True if any atom is aromatic

            if is_aromatic:
                # Generate the Murcko Scaffold, which represents the core structure
                scaffold = MurckoScaffold.GetScaffoldForMol(mol)
                scaffold_smiles = Chem.MolToSmiles(scaffold)  # Convert scaffold to SMILES

                # Store SMILES, scaffold, and aromaticity flag
                systems_homoleptic_aro.append([smiles_str, scaffold_smiles, 1])  # Aromaticity = 1
            else:
                # Non-aromatic case: store SMILES, no scaffold, and aromaticity flag
                systems_homoleptic_aro.append([smiles_str, 0, 0])  # Aromaticity = 0
        else:
            # Handle invalid SMILES strings
            print(f"Invalid SMILES string for index {i}.")

    # Step 2: Prepare scaffold and ligand fingerprints for PCA
    scaffold_descriptors = []  # Store fingerprints for scaffolds
    substituent_descriptors = []  # Store fingerprints for substituents

    for entry in systems_homoleptic_aro:
        smiles_str, scaffold_smiles, is_aromatic = entry

        # Generate Morgan fingerprint (radius 2, 1024-bit) for the ligand itself
        ligand_mol = Chem.MolFromSmiles(smiles_str)
        ligand_fp = rdMolDescriptors.GetMorganFingerprintAsBitVect(ligand_mol, 2, nBits=1024)
        ligand_fp = np.array(ligand_fp)  # Convert to NumPy array for calculations

        if is_aromatic:
            # Generate Morgan fingerprint for the Murcko Scaffold
            scaffold_mol = Chem.MolFromSmiles(scaffold_smiles)
            scaffold_fp = rdMolDescriptors.GetMorganFingerprintAsBitVect(scaffold_mol, 2, nBits=1024)
            scaffold_fp = np.array(scaffold_fp)  # Convert to NumPy array

            # Add scaffold fingerprint to the list for PCA
            scaffold_descriptors.append(scaffold_fp)

            # Calculate substituent fingerprint by subtracting scaffold from ligand
            substituent_fp = ligand_fp - scaffold_fp

            # Ensure the fingerprint is not zero; otherwise, set it to zero vector
            if np.all(substituent_fp == 0):
                substituent_fp = np.zeros_like(substituent_fp)

            # Add substituent fingerprint to the list for PCA
            substituent_descriptors.append(substituent_fp)
        else:
            # For non-aromatic molecules, store the ligand fingerprint directly
            substituent_descriptors.append(ligand_fp)

    # Step 3: Perform PCA on scaffold and substituent fingerprints
    scaffold_pca_values = compute_pca(scaffold_descriptors, n, debug=debug)
    substituent_pca_values = compute_pca(substituent_descriptors, n, debug=debug)

    # Step 4: Construct final descriptor array
    final_descriptors = []
    scaffold_index = 0  # Track index for scaffold PCA values

    # Loop over original systems to create descriptors
    for i, entry in enumerate(systems_homoleptic_aro):
        smiles_str, scaffold_smiles, is_aromatic = entry
        
        # Retrieve denticity information from the ligand
        denticity = systems_homoleptic[i].ligands[0].get_denticity()

        if is_aromatic:
            # Retrieve PCA values if aromatic
            murcko_id = scaffold_pca_values[scaffold_index]
            substituent_id = substituent_pca_values[i]
            scaffold_index += 1  # Move to the next scaffold value
        else:
            # For non-aromatic molecules, scaffold value is zero
            murcko_id = 0
            substituent_id = substituent_pca_values[i]

        # Create a descriptor array: [denticity, aromaticity flag, Murcko PCA, Substituent PCA]
        descriptor = [denticity, is_aromatic, murcko_id, substituent_id]
        final_descriptors.append(descriptor)  # Store the descriptor

    # Return the list of final descriptors for all systems
    return final_descriptors

In [8]:
#Test to figure out which value for n leads to pca_variance > 0.95 
test = compute_pca_descriptors(systems_homoleptic,150,debug=1)

Size of fingerprints array: 371
0.9893030672906961
Size of fingerprints array: 388
0.9795267025634633


In [9]:
def sorting_key(descriptor):
    return (descriptor[0], descriptor[1], descriptor[2], descriptor[3])

In [12]:
def sort_lists(PCA_descriptors_2D_w_distance, sorting_key1) :
    # Sort both lists
    sorted_indices = sorted(range(len(PCA_descriptors_2D_w_distance)), key=lambda i: sorting_key(PCA_descriptors_2D_w_distance[i]))
    # Apply the same sorting to pca_systems_homoleptic
    sorted_final_pca_descriptors = [PCA_descriptors_2D_w_distance[i] for i in sorted_indices]
    return sorted_indices, sorted_final_pca_descriptors

In [13]:
n = 200  # Number of PCA dimensions to evaluate
differences = []  # List to store the number of differences between sorted systems
total_systems = []  # List to store sorted homoleptic systems for each PCA dimension

# Loop through PCA dimensions from 1 to n in steps of 199 (evaluating 1D and 200D PCA)
for i in range(1, n + 1, 199):
    print("I'm doing", i)  # Informative print statement for current PCA dimension

    # Step 1: Compute PCA descriptors for the current dimension
    PCA_descriptors_iD_w_distance = compute_pca_descriptors(systems_homoleptic, i)

    # Step 2: Sort the systems based on the given sorting key
    sorted_indices, sorted_final_pca_descriptors = sort_lists(PCA_descriptors_iD_w_distance, sorting_key)

    # Step 3: Store PCA values and sorting indices as attributes of each ligand
    for idx, system in enumerate(systems_homoleptic):
        ligand = systems_homoleptic[idx].ligands[0]

        # Dynamically create attribute names for sorted index and PCA descriptor
        varname1 = f"sorted_index_{i}D_and_dist"
        setattr(ligand, varname1, sorted_indices[idx])  # Store sorted index

        varname2 = f"array_pca_morgan_fingerprint_{i}D_and_dist"
        setattr(ligand, varname2, PCA_descriptors_iD_w_distance[idx])  # Store PCA descriptor

    # Step 4: Store sorted systems based on current PCA dimension
    # Reorganize systems using sorted indices
    sorted_systems_homolepticiD = [systems_homoleptic[i] for i in sorted_indices]
    total_systems.append(sorted_systems_homolepticiD)  # Store sorted list for this dimension

# Step 5: Compare the order of sorted systems across different PCA dimensions
for i in range(len(total_systems) - 1):  # Compare successive PCA dimensions
    count_diffap_av = 0  # Initialize counter for different positions

    # Loop through all systems and compare sorted positions
    for j, system in enumerate(systems_homoleptic):
        # Retrieve reference codes from sorted systems for comparison
        name1 = total_systems[i][j].ligands[0].get_parent("cell").refcode
        name2 = total_systems[i + 1][j].ligands[0].get_parent("cell").refcode

        # Increment counter if systems are ordered differently
        if name1 != name2:
            count_diffap_av += 1

    # Store the number of different positions for the current comparison
    differences.append(count_diffap_av)
    print(count_diffap_av)  # Print the count for the current comparison

# Step 6: Print final list of differences
print(differences)

I'm doing 1
0.12778623161896074
0.07289260110701175
I'm doing 200
0.9999999999999996
0.9999656559105405
381
[381]


In [14]:
sorted_systems_homoleptic200D = total_systems[1] #Sorted systems for n=200
sorted_ligands200D = [sys.ligands[0].rdkit_obj for sys in sorted_systems_homoleptic200D]

In [15]:
mols2grid.display(sorted_ligands200D)

MolGridWidget()

In [ ]:
save_binary(systems_homoleptic, "systems_homoleptic_with_pcas.npy")